# 🏋️ Step 2: ペルソナ別 PPO 学習 → ONNX 生成

```
persona_rewards.json (Step1 で生成)
  ↓
ペルソナ A〜E を順番に PPO 学習 (各 ~60M steps)
  ↓
persona_A.onnx 〜 persona_E.onnx を単一ファイルで生成
  ↓
マイドライブ / mesa_persona_onnx / に保存
  ↓
step3_persona_city_sim.html で読み込む
```

## 修正済み問題
| 問題 | 対策 |
|------|------|
| `ModuleNotFoundError: onnxscript` | `onnxscript` をインストールに追加 |
| `LayerNormalization` opset 変換失敗 | `opset_version=17` に変更 |
| training mode 警告 | `model.eval()` + `torch.no_grad()` |
| `.onnx.data` 外部ファイル問題 | `onnx.save(save_as_external_data=False)` |
| GPU 使用率が低い | N_ENVS=4096, ROLLOUT=128, MINIBATCH=16384 に最適化 |

**Runtime: T4 GPU を選択してください**

In [ ]:
# ============================================================
# セル 1 置き換え — インストール (DINOv2追加)
# ============================================================
!pip install torch torchvision onnx onnxscript onnxruntime numpy -q
print('✓ Install complete')

In [ ]:
# ============================================================
# セル 2: インポート & GPU 確認 & Google Drive マウント
# ============================================================
import torch, torch.nn as nn, onnx
import numpy as np
import json, random, math, time, os, subprocess
from IPython.display import clear_output
import matplotlib.pyplot as plt, matplotlib

matplotlib.rcParams.update({
    'figure.facecolor': '#0a0c10', 'axes.facecolor': '#12161e',
    'text.color': '#c8d8e8',       'axes.labelcolor': '#c8d8e8',
    'xtick.color': '#4a5a6a',      'ytick.color': '#4a5a6a',
})

assert torch.cuda.is_available(), '❌ GPU 未検出。ランタイム → T4 GPU に変更してください'
DEVICE = torch.device('cuda')
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

def get_gpu_stats():
    try:
        r = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=2)
        v = r.stdout.strip().split(', ')
        return float(v[0]), float(v[1])/1024, float(v[2])/1024
    except:
        return 0.0, 0.0, 15.0

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/mesa_persona'       # persona_rewards.json の場所
ONNX_DIR = '/content/drive/MyDrive/mesa_persona_onnx'  # ONNX 出力先
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)
print(f'\n入力: {SAVE_DIR}')
print(f'出力: {ONNX_DIR}')

In [ ]:
# ============================================================
# セル 3 置き換え — 定数 (DINOv2版)
# ============================================================
OTHER=0; ROAD=1; BUILDING=2; TREE=3
GRID = 30

N_ENVS    = 4096
ROLLOUT   = 128
MINIBATCH = 16384
B         = N_ENVS * ROLLOUT

MAX_STEPS = 400
EPOCHS    = 4
CLIP_EPS  = 0.2
GAE_LAM   = 0.95
GAMMA     = 0.99
ENT_COEF  = 0.03
VF_COEF   = 0.5
LR        = 3e-4   # DINOv2 frozen の場合は少し大きめでOK
STEPS_PER_PERSONA = 60_000_000

MOVE_DIST = 0.25
ROT_RAD   = math.radians(20.0)

# ── FPV画像設定 (DINOv2版: 224×224) ──
IMG_W   = 224   # DINOv2 の標準入力サイズ
IMG_H   = 224
IMG_CH  = 3
OBS_DIM = IMG_CH * IMG_H * IMG_W   # 224*224*3 = 150528
ACT_DIM = 3

# ── DINOv2 設定 ──
DINO_MODEL   = 'dinov2_vits14'   # vits14=384次元 / vitb14=768次元
DINO_DIM     = 384               # vits14の出力次元
# DINO_MODEL = 'dinov2_vitb14'   # より高精度版
# DINO_DIM   = 768

# DINOv2 の正規化パラメータ (ImageNet統計)
DINO_MEAN = [0.485, 0.456, 0.406]
DINO_STD  = [0.229, 0.224, 0.225]

# レイキャスト設定 (FPV画像生成用)
RAY_MAX   = 8.0
N_RAYS    = IMG_W        # 224本
RAY_STEP  = 0.1
FOV       = math.radians(60.0)

# セルタイプの RGB色
CELL_RGB = torch.tensor([
    [ 12,  30,  74],
    [176, 180, 172],
    [196,  32,  32],
    [ 35, 104,  40],
], dtype=torch.float32, device='cpu') / 255.0
SKY_RGB   = torch.tensor([ 6, 12, 20], dtype=torch.float32) / 255.0
FLOOR_RGB = torch.tensor([26, 40, 32], dtype=torch.float32) / 255.0

RAY_ANGLES_T = torch.linspace(-FOV/2, FOV/2, IMG_W, dtype=torch.float32)
RAY_DISTS_T  = torch.arange(RAY_STEP, RAY_MAX+RAY_STEP, RAY_STEP, dtype=torch.float32)
N_STEPS_RAY  = len(RAY_DISTS_T)

print(f'N_ENVS:{N_ENVS}  B:{B:,}')
print(f'FP画像: {IMG_W}×{IMG_H}×{IMG_CH}ch (DINOv2標準サイズ)')
print(f'DINOv2: {DINO_MODEL}  出力次元: {DINO_DIM}')
print(f'バッファVRAM見積: {ROLLOUT*N_ENVS*OBS_DIM*4/1e9:.2f}GB')
print()
print('⚠️  224×224はバッファが大きい。VRAMが足りない場合:')
print('   N_ENVS=1024, ROLLOUT=64 に下げてください')

In [ ]:
# ============================================================
# セル 4: 有機的マップ生成 (Domain Randomization対応)
# ============================================================
from collections import deque

def make_map_organic(size=GRID, seed=None):
    """
    有機的都市マップ生成。
    seed=None のとき毎回違うマップ (Domain Randomization用)
    seed=42 など固定値のとき再現可能な固定マップ

    アルゴリズム:
      ① 格子道路を敷く
      ② ブロック内に建物・木を配置
      ③ 道路の区間セルをランダムに削除
         (道路ネットワークの連結性をBFSで保証)
      ④ 削除セルをTREE/OTHERで埋める
    """
    rng = random.Random(seed)  # seed=None → システム時刻でランダム

    grid = np.full((size, size), OTHER, dtype=np.int32)
    step = 4
    rows = list(range(0, size, step))
    cols = list(range(0, size, step))

    # ① 格子道路
    for r in rows: grid[r, :] = ROAD
    for c in cols: grid[:, c] = ROAD

    # ② ブロック内に建物・木を配置
    for ri in range(len(rows)-1):
        for ci in range(len(cols)-1):
            r0, r1 = rows[ri]+1, rows[ri+1]
            c0, c1 = cols[ci]+1, cols[ci+1]
            cells = [(r,c) for r in range(r0,r1) for c in range(c0,c1)]
            if not cells: continue
            b = rng.choice(cells)
            grid[b[0], b[1]] = BUILDING
            for r, c in cells:
                if (r,c) == b: continue
                v = rng.random()
                if   v < 0.25: grid[r,c] = TREE
                elif v < 0.45: grid[r,c] = BUILDING

    # 道路を最終確定 (建物が侵食しないように)
    for r in rows: grid[r, :] = ROAD
    for c in cols: grid[:, c] = ROAD

    # ③ 道路の連結性チェック関数
    #    「全道路セルが1つの連結成分か」を BFS で確認
    def road_connected(g):
        start = next(
            ((r,c) for r in range(size) for c in range(size) if g[r,c]==ROAD),
            None)
        if start is None: return True
        visited = set()
        q = deque([start])
        visited.add(start)
        while q:
            r, c = q.popleft()
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                if 0<=nr<size and 0<=nc<size and (nr,nc) not in visited \
                   and g[nr,nc]==ROAD:
                    visited.add((nr,nc))
                    q.append((nr,nc))
        # 全道路セルが訪問済みか
        return all(
            (r,c) in visited
            for r in range(size) for c in range(size)
            if g[r,c]==ROAD
        )

    # ③ 削除候補: 交差点以外の道路区間セル
    row_set = set(rows)
    col_set = set(cols)
    candidates = [
        (r, c)
        for r in range(size) for c in range(size)
        if grid[r,c] == ROAD
        and not (r in row_set and c in col_set)  # 交差点は除外
    ]
    rng.shuffle(candidates)

    # 削除率: 25〜50% をランダムに決定
    remove_ratio = 0.25 + rng.random() * 0.25
    max_remove   = int(len(candidates) * remove_ratio)
    removed      = 0

    for r, c in candidates:
        if removed >= max_remove: break
        grid[r, c] = OTHER  # 試しに削除
        if road_connected(grid):
            # 削除確定: TREE か OTHER で埋める
            grid[r, c] = TREE if rng.random() < 0.4 else OTHER
            removed += 1
        else:
            grid[r, c] = ROAD  # 連結が切れる → 復元

    return grid


# ── 初期マップ生成 (固定seed=42) ──
MAP_NP    = make_map_organic(seed=42)
MAP_T     = torch.tensor(MAP_NP, dtype=torch.long,  device=DEVICE)
MAP_F     = MAP_T.float()
PASS_T    = (MAP_T == ROAD) | (MAP_T == BUILDING)
BCELLS_NP = np.array([(r,c) for r in range(GRID)
                             for c in range(GRID) if MAP_NP[r,c]==BUILDING])
BCELLS_T  = torch.tensor(BCELLS_NP, dtype=torch.float32, device=DEVICE)
NB        = len(BCELLS_NP)

# ── マップ更新関数 (Domain Randomization用) ──
def randomize_map():
    """
    新しいランダムマップを生成して GPU テンソルを更新。
    PersonaVecEnv.step() の done 処理と組み合わせて呼ぶ。
    """
    global MAP_NP, MAP_T, MAP_F, PASS_T, BCELLS_NP, BCELLS_T, NB
    MAP_NP    = make_map_organic(seed=None)  # 毎回違うマップ
    MAP_T     = torch.tensor(MAP_NP, dtype=torch.long,  device=DEVICE)
    MAP_F     = MAP_T.float()
    PASS_T    = (MAP_T == ROAD) | (MAP_T == BUILDING)
    BCELLS_NP = np.array([(r,c) for r in range(GRID)
                                 for c in range(GRID) if MAP_NP[r,c]==BUILDING])
    BCELLS_T  = torch.tensor(BCELLS_NP, dtype=torch.float32, device=DEVICE)
    NB        = len(BCELLS_NP)


# ── 初期マップを可視化 ──
cmap = matplotlib.colors.ListedColormap(['#1a3a6a','#d0d0c8','#c42020','#256b28'])
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0a0c10')
for i, ax in enumerate(axes):
    test_map = make_map_organic(seed=i*100 if i > 0 else 42)
    ax.imshow(test_map, cmap=cmap, vmin=0, vmax=3, origin='upper')
    roads = int((test_map==ROAD).sum())
    bldgs = int((test_map==BUILDING).sum())
    ax.set_title(
        f'seed={42 if i==0 else i*100}  roads:{roads}  bldgs:{bldgs}',
        color='#c8d8e8', fontsize=9)
    ax.axis('off')
plt.suptitle('有機的マップのサンプル (3パターン)', color='#00ddb4', fontsize=12)
plt.tight_layout()
plt.show()

print(f'初期マップ: Buildings={NB}  MAP_T={MAP_T.device}')
print(f'有機的マップ生成 ✓  randomize_map() で学習中に更新可能')

In [ ]:
# ============================================================
# セル 5 置き換え — FP画像レンダリング (GPU一括)
# ============================================================
@torch.jit.script
def render_fp_batch(
    xs: torch.Tensor,    # (N,) エージェントrow座標
    ys: torch.Tensor,    # (N,) エージェントcol座標
    ths: torch.Tensor,   # (N,) 向き (ラジアン)
    map_t: torch.Tensor, # (GRID, GRID) セルタイプ
    cell_rgb: torch.Tensor,  # (4, 3) セルタイプ別RGB
    sky_rgb: torch.Tensor,   # (3,) 空の色
    floor_rgb: torch.Tensor, # (3,) 地面の色
    ray_angles: torch.Tensor,# (W,) レイ角度
    ray_dists: torch.Tensor, # (S,) サンプル距離
    img_w: int, img_h: int, grid: int, ray_max: float,
) -> torch.Tensor:
    """
    N エージェント分の FP 画像を GPU で一括レンダリング。
    returns: (N, 3, H, W)  float32  [0, 1]
    """
    N = xs.shape[0]
    W = img_w; H = img_h; S = ray_dists.shape[0]

    # 各レイの方向 (N, W)
    angles = ths.unsqueeze(1) + ray_angles.unsqueeze(0)  # (N, W)
    dx = torch.cos(angles)  # (N, W)
    dy = torch.sin(angles)  # (N, W)

    # サンプル点 (N, W, S)
    px = xs[:,None,None] + dx[:,:,None] * ray_dists[None,None,:]  # (N,W,S)
    py = ys[:,None,None] + dy[:,:,None] * ray_dists[None,None,:]  # (N,W,S)

    # 範囲チェック & セルタイプ取得
    in_bounds = (px>=0.0)&(px<float(grid))&(py>=0.0)&(py<float(grid))
    ri = px.long().clamp(0, grid-1)
    ci = py.long().clamp(0, grid-1)
    ct = map_t[ri, ci]  # (N, W, S)  セルタイプ
    ct = torch.where(in_bounds, ct, torch.ones_like(ct))  # 範囲外→ROAD(1)

    # ROAD 以外でヒット
    is_wall = ct != 1  # ROAD=1
    first   = torch.argmax(is_wall.long(), dim=2)       # (N, W)
    has_hit = is_wall.any(dim=2)                         # (N, W)
    first   = torch.where(has_hit, first,
                          torch.full_like(first, S-1))   # ヒットなし→最大距離

    hit_ct   = ct.gather(2, first.unsqueeze(2)).squeeze(2)   # (N, W) セルタイプ
    hit_dist = ray_dists[first]                               # (N, W) 距離

    # 道路方向はヒットなし扱い
    hit_ct = torch.where(has_hit, hit_ct, torch.zeros_like(hit_ct))

    # 距離による明るさ (近いほど明るい)
    brightness = (1.0 - (hit_dist / ray_max)).clamp(0.15, 1.0)  # (N, W)

    # セルタイプ→RGB (N, W, 3)
    wall_rgb = cell_rgb[hit_ct.long()]  # (N, W, 3)
    wall_rgb = wall_rgb * brightness.unsqueeze(2)  # 距離で暗くする

    # 柱の高さ: 距離に反比例 (DOOM式)
    # 道路方向(has_hit=False) の柱高さ = 0
    safe_dist   = hit_dist.clamp(min=0.1)
    col_height  = torch.where(
        has_hit,
        (float(H) * 1.5 / safe_dist).clamp(0.0, float(H)),
        torch.zeros_like(hit_dist)
    )  # (N, W)

    # ── 画像を描画 ──
    # まず背景 (空 + 地面) を敷く
    img = torch.zeros(N, 3, H, W, dtype=torch.float32,
                      device=xs.device)  # (N, 3, H, W)

    # 行インデックス (H,)
    row_idx = torch.arange(H, dtype=torch.float32, device=xs.device)

    # 空 (上半分)
    sky_mask = (row_idx < float(H) * 0.5).float()  # (H,)
    for c in range(3):
        img[:, c, :, :] += sky_rgb[c]   * sky_mask[None, :, None]
        img[:, c, :, :] += floor_rgb[c] * (1.0 - sky_mask[None, :, None])

    # 柱を描画 (各列ごと)
    # col_height: (N, W)  wall_rgb: (N, W, 3)
    y0 = ((float(H) - col_height) * 0.5).long().clamp(0, H-1)  # (N, W)
    y1 = (y0 + col_height.long()).clamp(0, H)                    # (N, W)

    # ピクセルごとにマスクを作成して描画
    # row_mask: 各列で [y0, y1) に1が立つマスク (N, W, H)
    rows = torch.arange(H, device=xs.device)[None, None, :]  # (1,1,H)
    in_col = (rows >= y0[:, :, None]) & (rows < y1[:, :, None])  # (N,W,H)

    # (N, W, H) → (N, H, W) に転置して画像に書き込む
    in_col_t = in_col.permute(0, 2, 1).float()  # (N, H, W)

    for c in range(3):
        # wall_rgb[:,:,c]: (N, W) → (N, 1, W) にブロードキャスト
        wc = wall_rgb[:, :, c].unsqueeze(1)  # (N, 1, W)
        img[:, c, :, :] = torch.where(
            in_col_t > 0,
            wc.expand_as(img[:, c, :, :]),
            img[:, c, :, :]
        )

    return img.clamp(0.0, 1.0)


# ── GPU用テンソルを作成 ──
CELL_RGB_GPU  = CELL_RGB.to(DEVICE)
SKY_RGB_GPU   = SKY_RGB.to(DEVICE)
FLOOR_RGB_GPU = FLOOR_RGB.to(DEVICE)
RAY_ANGLES_GPU= RAY_ANGLES_T.to(DEVICE)
RAY_DISTS_GPU = RAY_DISTS_T.to(DEVICE)

# warm-up
_x=torch.rand(N_ENVS,device=DEVICE)*GRID
_y=torch.rand(N_ENVS,device=DEVICE)*GRID
_t=torch.rand(N_ENVS,device=DEVICE)*math.pi*2
for _ in range(3):
    render_fp_batch(
        _x,_y,_t,MAP_T,CELL_RGB_GPU,SKY_RGB_GPU,FLOOR_RGB_GPU,
        RAY_ANGLES_GPU,RAY_DISTS_GPU,IMG_W,IMG_H,GRID,RAY_MAX)
torch.cuda.synchronize()
t0=time.time()
for _ in range(20):
    render_fp_batch(
        _x,_y,_t,MAP_T,CELL_RGB_GPU,SKY_RGB_GPU,FLOOR_RGB_GPU,
        RAY_ANGLES_GPU,RAY_DISTS_GPU,IMG_W,IMG_H,GRID,RAY_MAX)
torch.cuda.synchronize()
print(f'render_fp_batch JIT: {(time.time()-t0)/20*1000:.1f}ms/call ({N_ENVS} envs)')

# サンプル画像を可視化
sample = render_fp_batch(
    _x[:4],_y[:4],_t[:4],MAP_T,CELL_RGB_GPU,SKY_RGB_GPU,FLOOR_RGB_GPU,
    RAY_ANGLES_GPU,RAY_DISTS_GPU,IMG_W,IMG_H,GRID,RAY_MAX
).cpu()
fig,axes=plt.subplots(1,4,figsize=(12,3))
fig.patch.set_facecolor('#0a0c10')
for i,ax in enumerate(axes):
    ax.imshow(sample[i].permute(1,2,0).numpy())
    ax.set_title(f'Agent {i}',color='#c8d8e8',fontsize=9)
    ax.axis('off')
plt.suptitle(f'FP画像サンプル ({IMG_W}×{IMG_H}×{IMG_CH}ch)',
             color='#00ddb4',fontsize=11)
plt.tight_layout(); plt.show()
del _x,_y,_t
print('render_fp_batch ✓')

In [ ]:
# ============================================================
# セル 6 置き換え — PersonaVecEnv (FP画像観測版)
# ============================================================
class PersonaVecEnv:
    def __init__(self, n, rp):
        self.n=n; self.rp=rp
        self.x    =torch.zeros(n,device=DEVICE); self.y    =torch.zeros(n,device=DEVICE)
        self.th   =torch.zeros(n,device=DEVICE)
        self.gx   =torch.zeros(n,device=DEVICE); self.gy   =torch.zeros(n,device=DEVICE)
        self.pdist=torch.zeros(n,device=DEVICE)
        self.stall=torch.zeros(n,dtype=torch.long,device=DEVICE)
        self.scnt =torch.zeros(n,dtype=torch.long,device=DEVICE)
        self.px   =torch.zeros(n,device=DEVICE); self.py   =torch.zeros(n,device=DEVICE)
        self.visited=torch.zeros(n,GRID,GRID,dtype=torch.uint8,device=DEVICE)
        self._rst(torch.ones(n,dtype=torch.bool,device=DEVICE))

    def _rand_b(self,mask):
        idx=torch.randint(0,NB,(self.n,),device=DEVICE)
        return BCELLS_T[idx,0]+0.5, BCELLS_T[idx,1]+0.5

    def _dist(self): return torch.sqrt((self.x-self.gx)**2+(self.y-self.gy)**2)

    def _rst(self,mask):
        bx,by=self._rand_b(mask); gx,gy=self._rand_b(mask)
        th=torch.rand(self.n,device=DEVICE)*math.pi*2
        nd=torch.sqrt((bx-gx)**2+(by-gy)**2)
        self.x    =torch.where(mask,bx,self.x);   self.y    =torch.where(mask,by,self.y)
        self.th   =torch.where(mask,th,self.th)
        self.gx   =torch.where(mask,gx,self.gx);  self.gy   =torch.where(mask,gy,self.gy)
        self.pdist=torch.where(mask,nd,self.pdist)
        self.stall=torch.where(mask,torch.zeros_like(self.stall),self.stall)
        self.scnt =torch.where(mask,torch.zeros_like(self.scnt), self.scnt)
        self.px   =torch.where(mask,bx,self.px);  self.py   =torch.where(mask,by,self.py)
        self.visited[mask]=0

    def reset_all(self):
        self._rst(torch.ones(self.n,dtype=torch.bool,device=DEVICE))
        return self._obs()

    def _obs(self):
        """FP画像を GPU で一括生成。returns: (N, 3, H, W)"""
        return render_fp_batch(
            self.x, self.y, self.th,
            MAP_T, CELL_RGB_GPU, SKY_RGB_GPU, FLOOR_RGB_GPU,
            RAY_ANGLES_GPU, RAY_DISTS_GPU,
            IMG_W, IMG_H, GRID, RAY_MAX
        )

    def step(self, actions):
        rp  = self.rp
        rew = torch.full((self.n,), -rp['step_penalty'], device=DEVICE)
        self.px=self.x.clone(); self.py=self.y.clone()
        rot=torch.where(actions==1,torch.full_like(self.th,-ROT_RAD),
            torch.where(actions==2,torch.full_like(self.th, ROT_RAD),
                        torch.zeros_like(self.th)))
        self.th=(self.th+rot)%(math.pi*2)
        fwd=(actions==0)
        nx=(self.x+torch.cos(self.th)*MOVE_DIST).clamp(0.01,GRID-0.01)
        ny=(self.y+torch.sin(self.th)*MOVE_DIST).clamp(0.01,GRID-0.01)
        ri=nx.long().clamp(0,GRID-1); ci=ny.long().clamp(0,GRID-1)
        passable=PASS_T[ri,ci]
        can_move=fwd&passable; blocked=fwd&~passable
        self.x=torch.where(can_move,nx,self.x); self.y=torch.where(can_move,ny,self.y)
        ri2=self.x.long().clamp(0,GRID-1); ci2=self.y.long().clamp(0,GRID-1)
        cur_cell=MAP_T[ri2,ci2]
        rew=torch.where(can_move&(cur_cell==ROAD),  rew+rp['road_bonus'],      rew)
        rew=torch.where(cur_cell==BUILDING,          rew+rp['building_bonus'],  rew)
        rew=torch.where(fwd,                         rew+rp['forward_bias'],    rew)
        rew=torch.where(blocked,                     rew-rp['violation_penalty'],rew)
        env_idx=torch.arange(self.n,device=DEVICE)
        was=self.visited[env_idx,ri2,ci2].bool()
        rew=torch.where(can_move&~was, rew+rp['explore_bonus'],   rew)
        rew=torch.where(can_move& was, rew-rp['revisit_penalty'], rew)
        self.visited[env_idx,ri2,ci2]=1
        if rp['social_bonus']>0:
            ox=torch.roll(self.x,1); oy=torch.roll(self.y,1)
            near=(torch.sqrt((self.x-ox)**2+(self.y-oy)**2)<2.0)&can_move
            rew=torch.where(near,rew+rp['social_bonus'],rew)
        moved=(torch.abs(self.x-self.px)+torch.abs(self.y-self.py))>0.05
        self.stall=torch.where(moved,torch.zeros_like(self.stall),self.stall+1)
        rew=torch.where(self.stall>3,rew-rp['stall_penalty'],rew)
        nd=self._dist()
        rew=torch.where(nd<self.pdist,rew+0.5,rew)
        rew=torch.where(nd>self.pdist,rew-0.2,rew)
        self.pdist=nd
        arrived=(cur_cell==BUILDING)&(nd<0.8)
        rew=torch.where(arrived,rew+rp['goal_reward'],rew)
        ngx,ngy=self._rand_b(arrived)
        self.gx=torch.where(arrived,ngx,self.gx); self.gy=torch.where(arrived,ngy,self.gy)
        self.pdist=torch.where(arrived,
            torch.sqrt((self.x-self.gx)**2+(self.y-self.gy)**2),self.pdist)
        self.scnt+=1
        done=self.scnt>=MAX_STEPS
        self._rst(done)
        return self._obs(),rew,done

print('PersonaVecEnv (FP画像版) ✓')

In [ ]:
# ============================================================
# セル 7 置き換え — DINOv2 PolicyNet
# ============================================================
import torchvision.transforms as T

# DINOv2 を torch.hub からロード
print(f'DINOv2 ({DINO_MODEL}) をロード中...')
dino_backbone = torch.hub.load(
    'facebookresearch/dinov2',
    DINO_MODEL,
    pretrained=True,
)
dino_backbone = dino_backbone.to(DEVICE)
dino_backbone.eval()

# 全パラメータを frozen (勾配なし)
for param in dino_backbone.parameters():
    param.requires_grad = False

n_dino = sum(p.numel() for p in dino_backbone.parameters())
print(f'✓ DINOv2 loaded: {n_dino:,} params (全frozen)')

# ImageNet 正規化 (GPU上で実行)
dino_mean = torch.tensor(DINO_MEAN, device=DEVICE).view(1,3,1,1)
dino_std  = torch.tensor(DINO_STD,  device=DEVICE).view(1,3,1,1)

def normalize_for_dino(imgs: torch.Tensor) -> torch.Tensor:
    """
    FPV画像 [0,1] を DINOv2 用に ImageNet 正規化する。
    imgs: (N, 3, H, W)  float32  [0, 1]
    returns: (N, 3, H, W) 正規化済み
    """
    return (imgs - dino_mean) / dino_std


# ── 建物分類ヘッドを ONNX から読み込む ──
import onnxruntime as ort_clf

CLASSIFIER_PATH = f'{ONNX_DIR}/building_classifier.onnx'
CLASSIFIER_META  = f'{ONNX_DIR}/building_classifier_meta.json'

if os.path.exists(CLASSIFIER_PATH):
    clf_sess = ort_clf.InferenceSession(
        CLASSIFIER_PATH, providers=['CPUExecutionProvider'])
    with open(CLASSIFIER_META) as f:
        clf_meta = json.load(f)
    N_BLDG_CLASSES = clf_meta['n_classes']
    BLDG_CLASSES   = clf_meta['classes']
    print(f'✓ 建物分類ヘッド読み込み: {N_BLDG_CLASSES}クラス  精度={clf_meta.get("best_val_acc",0):.1%}')
    print(f'  クラス: {BLDG_CLASSES}')
else:
    # 分類ヘッドなし → ゼロベクトルで代替
    clf_sess       = None
    N_BLDG_CLASSES = 8
    BLDG_CLASSES   = ['gyudon','ramen','bento','cafe','office','house','conbini','hospital']
    print('⚠ building_classifier.onnx が見つかりません')
    print('  step1_5_building_classifier.ipynb を実行してください')
    print('  → 分類スコアなしで学習を続けます')


def classify_buildings_batch(feats_np: np.ndarray) -> np.ndarray:
    """
    DINOv2 特徴量から建物クラス確率を計算 (CPU・バッチ処理)
    feats_np: (N, DINO_DIM)
    returns:  (N, N_BLDG_CLASSES)  softmax確率
    """
    if clf_sess is None:
        return np.zeros((len(feats_np), N_BLDG_CLASSES), dtype=np.float32)
    logits = clf_sess.run(None, {'dino_feat': feats_np})[0]  # (N, C)
    exp    = np.exp(logits - logits.max(axis=1, keepdims=True))
    return (exp / exp.sum(axis=1, keepdims=True)).astype(np.float32)


class PolicyNet(nn.Module):
    """
    DINOv2 + 建物分類ヘッド を組み込んだ Actor-Critic。

    入力: (N, 3, 224, 224) FPV画像  または  (N, 3*224*224) フラット
    処理:
      DINOv2 (frozen)         → CLS token (N, DINO_DIM=384)
      BuildingClassifier(onnx)→ クラス確率 (N, N_BLDG_CLASSES=8)
      concat                  → (N, 384+8=392)
      FC ヘッド               → (N, 256)
      Actor/Critic            → logits(N,3) / value(N,1)

    「今見えている建物が何屋か」という情報がポリシーに伝わる。
    目的クラス(牛丼屋)を報酬関数で指定すれば、
    その建物を優先して訪れるポリシーが学習される。
    """
    def __init__(self):
        super().__init__()
        # FC ヘッド (DINOv2 + 分類スコア → 256)
        self.fc = nn.Sequential(
            nn.Linear(DINO_DIM + N_BLDG_CLASSES, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
        )
        self.actor  = nn.Linear(256, ACT_DIM)
        self.critic = nn.Linear(256, 1)

    def _encode(self, x: torch.Tensor) -> tuple[torch.Tensor, np.ndarray]:
        """
        FPV画像 → (DINOv2特徴, 建物分類スコア)
        returns: feat (N, DINO_DIM) GPU,  bldg_probs (N, C) numpy
        """
        if x.dim() == 2:
            x = x.view(-1, IMG_CH, IMG_H, IMG_W)
        x = normalize_for_dino(x)
        with torch.no_grad():
            feat = dino_backbone(x)          # (N, 384) GPU
        feats_np = feat.cpu().numpy().astype(np.float32)
        bldg_probs = classify_buildings_batch(feats_np)  # (N, C) numpy
        return feat, bldg_probs

    def forward(self, x: torch.Tensor):
        feat, bldg_probs = self._encode(x)
        bldg_t = torch.tensor(bldg_probs, device=feat.device)  # (N, C)
        combined = torch.cat([feat, bldg_t], dim=1)             # (N, 384+8)
        h = self.fc(combined)
        return self.actor(h), self.critic(h)

    def act(self, x: torch.Tensor):
        lg, val = self.forward(x)
        dist    = torch.distributions.Categorical(torch.softmax(lg, -1))
        a       = dist.sample()
        return a, dist.log_prob(a), dist.entropy(), val.squeeze(-1)

    def get_building_probs(self, x: torch.Tensor) -> np.ndarray:
        """現在の視野の建物分類確率を取得 (ログ用)"""
        _, bldg_probs = self._encode(x)
        return bldg_probs


# ── 動作確認 ──
n_fc = sum(p.numel() for p in PolicyNet().parameters())
print(f'PolicyNet FC部分 : {n_fc:,} params (これだけ学習)')
print(f'DINOv2 frozen    : {n_dino:,} params')
print(f'分類ヘッド       : {"あり" if clf_sess else "なし (ゼロ代替)"}')
print()

# 推論テスト
_net   = PolicyNet().to(DEVICE)
_dummy = torch.randn(2, IMG_CH, IMG_H, IMG_W, device=DEVICE)
_lg, _val = _net.forward(_dummy)
assert _lg.shape == (2, ACT_DIM)
print(f'forward OK: logits={_lg.shape}  val={_val.shape}')

# 建物分類スコアのテスト
_bldg = _net.get_building_probs(_dummy)
print(f'building_probs: {_bldg.shape}  (各クラスの確率)')
print(f'  {dict(zip(BLDG_CLASSES, _bldg[0].round(3)))}')

util, used, total = get_gpu_stats()
print(f'VRAM: {used:.2f}/{total:.1f}GB')
del _net, _dummy, _lg, _val, _bldg
print('✓ PolicyNet DINOv2+分類ヘッド版 OK')

In [ ]:
# ============================================================
# セル 8 置き換え — ONNX エクスポート (FP画像CNN版)
# ============================================================
def export_persona_onnx(policy, persona_id, rp):
    policy.eval()
    policy_cpu = policy.to('cpu')

    # DINOv2 frozen部分は除き、FC+Actorヘッドだけexport
    # HTML側では別途DINOv2相当の特徴量を渡す
    # DINOv2 frozen部分を除き FC+Actor だけ export
    class FCActorOnly(nn.Module):
        def __init__(self, p): super().__init__(); self.fc=p.fc; self.actor=p.actor
        def forward(self, feat): return self.actor(self.fc(feat))

    actor = FCActorOnly(policy_cpu); actor.eval()
    tmp_path  = f'/tmp/persona_{persona_id}_tmp.onnx'
    onnx_path = f'{ONNX_DIR}/persona_{persona_id}.onnx'

    # フラット形式でexport (HTMLでのfetch推論に対応)
    dummy_flat = torch.zeros(1, DINO_DIM)  # DINOv2: FC入力は特徴ベクトル

    with torch.no_grad():
        torch.onnx.export(
            actor, dummy_flat, tmp_path,
            input_names=['state'], output_names=['logits'],
            dynamic_axes={'state':{0:'batch'}, 'logits':{0:'batch'}},
            opset_version=17,
        )

    onnx_model = onnx.load(tmp_path)
    onnx.save(onnx_model, onnx_path, save_as_external_data=False)
    for f in [tmp_path, tmp_path+'.data']:
        if os.path.exists(f): os.remove(f)

    size_mb = os.path.getsize(onnx_path)/1e6
    print(f'  ✓ persona_{persona_id}.onnx  ({size_mb:.1f}MB)')

    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    dummy_np = np.zeros((1, DINO_DIM), dtype=np.float32)
    logits = sess.run(None, {'state': dummy_np})[0]
    print(f'  ✓ 推論テスト OK  logits={logits[0].round(3)}')

    meta = {
        'persona_id':   persona_id,
        'persona_name': rp.get('persona_name', f'Persona {persona_id}'),
        'description':  rp.get('description', ''),
        'reward_params': rp,
        'obs_type':     'fp_image',  # ← 観測タイプを明記
        'input_size':   DINO_DIM,   # DINOv2 CLS token 次元
        'img_w': IMG_W, 'img_h': IMG_H, 'img_ch': IMG_CH,
        'fov_deg': 60.0, 'ray_max': RAY_MAX,
        'output_size':  ACT_DIM,
        'actions':      ['forward', 'turn_left', 'turn_right'],
        'move_dist':    MOVE_DIST, 'rot_deg': 20.0,
        'grid_size':    GRID,
        'map':          MAP_NP.tolist(),
        'building_cells': BCELLS_NP.tolist(),
        'cell_rgb': [
            [12,30,74], [176,180,172], [196,32,32], [35,104,40]
        ],
        'sky_rgb':   [6,12,20],
        'floor_rgb': [26,40,32],
    }
    meta_path = onnx_path.replace('.onnx', '_meta.json')
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    print(f'  ✓ persona_{persona_id}_meta.json 保存')

    policy.to(DEVICE)
    return onnx_path

print('export_persona_onnx (FP-CNN版) ✓')

In [ ]:
# ============================================================
# セル 9 置き換え — train_persona (FP画像CNN版)
# バッファを (ROLLOUT, N, 3, H, W) の5次元に変更
# ============================================================
MAP_RANDOMIZE_EVERY = 20

def train_persona(persona_id, rp, total_steps=STEPS_PER_PERSONA):
    print('='*60)
    print(f'学習開始: [{persona_id}] {rp["persona_name"]}')
    print(f'  観測: FP画像 {IMG_W}×{IMG_H}×{IMG_CH}ch')
    print(f'  目標: {total_steps/1e6:.0f}M steps')
    dr_str = f'{MAP_RANDOMIZE_EVERY} update ごと' if MAP_RANDOMIZE_EVERY > 0 else '無効'
    print(f'  Domain Randomization: {dr_str}')
    print('='*60)

    policy    = PolicyNet().to(DEVICE)
    optimizer = torch.optim.Adam(policy.parameters(), lr=LR)
    env       = PersonaVecEnv(N_ENVS, rp)

    ckpt_path    = f'{SAVE_DIR}/ckpt_{persona_id}.pt'
    start_update = 0
    log = {'reward':[], 'trips':[], 'viols':[], 'explore':[],
           'gpu_util':[], 'gpu_vram':[], 'map_changes':[]}

    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        policy.load_state_dict(ck['policy'])
        optimizer.load_state_dict(ck['optimizer'])
        start_update = ck.get('update', 0)
        log = ck.get('log', log)
        print(f'  ✓ 再開: update={start_update}')

    obs = env.reset_all()  # (N, 3, H, W)

    # ── バッファ: 画像を5次元で保持 ──
    obs_buf  = torch.zeros(ROLLOUT, N_ENVS, IMG_CH, IMG_H, IMG_W, device=DEVICE)
    act_buf  = torch.zeros(ROLLOUT, N_ENVS, dtype=torch.long, device=DEVICE)
    rew_buf  = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)
    done_buf = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)
    logp_buf = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)
    val_buf  = torch.zeros(ROLLOUT, N_ENVS, device=DEVICE)

    total = start_update * B
    r_s = t_s = v_s = e_s = 0.0
    t0  = time.time()
    LOG_EVERY   = 10
    map_changes = 0

    for update in range(start_update, total_steps // B + 1):

        # Domain Randomization
        if MAP_RANDOMIZE_EVERY > 0 and update > 0 and update % MAP_RANDOMIZE_EVERY == 0:
            randomize_map()
            env.visited.zero_()
            obs = env.reset_all()
            map_changes += 1

        # Rollout
        for t in range(ROLLOUT):
            with torch.no_grad():
                actions, logps, _, values = policy.act(obs)
            obs_buf[t]  = obs
            act_buf[t]  = actions
            logp_buf[t] = logps
            val_buf[t]  = values
            obs, rew, done = env.step(actions)
            rew_buf[t]  = rew
            done_buf[t] = done.float()
            total += N_ENVS
            r_s += rew.sum().item()
            t_s += (rew >= rp['goal_reward']*0.8).sum().item()
            v_s += (rew <= -rp['violation_penalty']*0.8).sum().item()
            e_s += (rew >= rp['explore_bonus']*0.8).sum().item()

        # GAE
        with torch.no_grad():
            _, lv = policy.forward(obs); lv = lv.squeeze(-1)
        adv = torch.zeros_like(rew_buf)
        gae = torch.zeros(N_ENVS, device=DEVICE)
        for t in reversed(range(ROLLOUT)):
            nv    = lv if t==ROLLOUT-1 else val_buf[t+1]
            delta = rew_buf[t] + GAMMA*nv*(1-done_buf[t]) - val_buf[t]
            gae   = delta + GAMMA*GAE_LAM*(1-done_buf[t])*gae
            adv[t] = gae
        ret = adv + val_buf

        # PPO — reshape で5次元→4次元 (B, C, H, W)
        obs_f  = obs_buf.reshape(B, IMG_CH, IMG_H, IMG_W)
        act_f  = act_buf.reshape(B)
        logp_f = logp_buf.reshape(B)
        adv_f  = adv.reshape(B)
        ret_f  = ret.reshape(B)
        adv_f  = (adv_f - adv_f.mean()) / (adv_f.std() + 1e-8)

        for _ in range(EPOCHS):
            perm = torch.randperm(B, device=DEVICE)
            for s in range(0, B, MINIBATCH):
                mb     = perm[s:s+MINIBATCH]
                lg, vs = policy.forward(obs_f[mb])
                dist   = torch.distributions.Categorical(torch.softmax(lg,-1))
                nlp    = dist.log_prob(act_f[mb])
                ent    = dist.entropy().mean()
                ratio  = torch.exp(nlp - logp_f[mb])
                lpi    = -torch.min(
                    ratio*adv_f[mb],
                    torch.clamp(ratio,1-CLIP_EPS,1+CLIP_EPS)*adv_f[mb]
                ).mean()
                lvf    = (vs.squeeze()-ret_f[mb]).pow(2).mean()
                loss   = lpi + VF_COEF*lvf - ENT_COEF*ent
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
                optimizer.step()

        # ログ
        if update % LOG_EVERY == 0 and update > 0:
            d  = LOG_EVERY * B
            ar = r_s/d; at = t_s/d*100; av = v_s/d*100; ae = e_s/d*100
            gpu_util, gpu_used, _ = get_gpu_stats()
            sps = LOG_EVERY*B/(time.time()-t0)
            log['reward'].append(round(float(ar),4))
            log['trips'].append(round(float(at),3))
            log['viols'].append(round(float(av),3))
            log['explore'].append(round(float(ae),3))
            log['gpu_util'].append(round(float(gpu_util),1))
            log['gpu_vram'].append(round(float(gpu_used),3))
            log['map_changes'].append(map_changes)
            r_s=t_s=v_s=e_s=0.0; t0=time.time()
            clear_output(wait=True)

            fig = plt.figure(figsize=(22,3.5))
            fig.patch.set_facecolor('#0a0c10')
            for i,(k,c,title) in enumerate([
                ('reward','#00ddb4','Reward/step'),
                ('trips','#ffc840','GoalRate%'),
                ('viols','#ff5050','ViolRate%'),
                ('explore','#aa88ff','ExploreRate%'),
                ('gpu_util','#44aaff','GPU%'),
            ]):
                ax=fig.add_subplot(1,6,i+1); ax.set_facecolor('#12161e')
                v=log[k]
                if len(v)>=2: ax.plot(v,color=c,lw=1.5)
                ax.set_title(f'{title}\n{v[-1]:.2f}' if v else title,
                             color='#c8d8e8',fontsize=8)
                ax.grid(color='#1e2530',lw=0.5); ax.spines[:].set_color('#1e2530')
                if k=='gpu_util': ax.set_ylim(0,100)
            # 現在のFPサンプル画像
            ax_img=fig.add_subplot(1,6,6)
            with torch.no_grad():
                sample=render_fp_batch(
                    env.x[:1],env.y[:1],env.th[:1],
                    MAP_T,CELL_RGB_GPU,SKY_RGB_GPU,FLOOR_RGB_GPU,
                    RAY_ANGLES_GPU,RAY_DISTS_GPU,
                    IMG_W,IMG_H,GRID,RAY_MAX
                ).cpu()
            ax_img.imshow(sample[0].permute(1,2,0).numpy())
            ax_img.set_title('現在のFP観測', color='#c8d8e8', fontsize=8)
            ax_img.axis('off')
            fig.suptitle(
                f'[{persona_id}] {rp["persona_name"]}  |  '
                f'Update:{update:,}  Steps:{total/1e6:.1f}M  '
                f'({total/total_steps*100:.1f}%)  Maps:{map_changes}',
                color='#00ddb4', fontsize=11)
            plt.tight_layout(); plt.show()
            print(f'[{persona_id}] Upd:{update:5d} | R:{ar:7.4f} | '
                  f'Goal:{at:.2f}% | Viol:{av:.2f}% | Exp:{ae:.2f}% | '
                  f'{sps:.0f}sps | GPU:{gpu_util:.0f}% VRAM:{gpu_used:.2f}GB')

        if update%50==0 and update>0:
            torch.save({'policy':policy.state_dict(),
                        'optimizer':optimizer.state_dict(),
                        'update':update,'log':log,
                        'map_changes':map_changes},ckpt_path)
            print(f'  💾 [{persona_id}] ckpt 保存 (update={update})')

    print(f'\n[{persona_id}] 学習完了 ({total/1e6:.1f}M steps)')
    return policy, log

print('train_persona (FP-CNN版) ✓')

In [ ]:
# ============================================================
# セル 10: 報酬パラメータ読み込み
# ============================================================
rewards_path = f'{SAVE_DIR}/persona_rewards.json'
assert os.path.exists(rewards_path), (
    f'❌ {rewards_path} が見つかりません。\n'
    f'   step1_persona_reward_gen.ipynb を先に実行するか、\n'
    f'   persona_rewards.json (サンプル) を {SAVE_DIR} にアップロードしてください。'
)

with open(rewards_path, encoding='utf-8') as f:
    all_rewards = json.load(f)

# 数値を float に変換
FLOAT_KEYS = ['explore_bonus','building_bonus','road_bonus','forward_bias',
               'revisit_penalty','violation_penalty','goal_reward',
               'step_penalty','social_bonus','stall_penalty']
for pid,rp in all_rewards.items():
    for k in FLOAT_KEYS:
        rp[k] = float(rp.get(k, 0.0))

print(f'✓ {len(all_rewards)} ペルソナ読み込み完了')
print()
for pid,rp in all_rewards.items():
    print(f'  [{pid}] {rp["persona_name"]}')
    print(f'       explore={rp["explore_bonus"]:.1f}  '
          f'goal={rp["goal_reward"]:.1f}  '
          f'social={rp["social_bonus"]:.1f}  '
          f'step_pen={rp["step_penalty"]:.2f}')

In [ ]:
# ============================================================
# セル 11: 全ペルソナを順次学習 → ONNX エクスポート
# ============================================================
# 学習するペルソナを選択
# 全ペルソナ: list(all_rewards.keys())
# 途中から:   ['B','C','D','E']  など
TRAIN_PERSONAS = list(all_rewards.keys())

onnx_paths = {}

for pid in TRAIN_PERSONAS:
    rp = all_rewards[pid]

    # 学習
    policy, log = train_persona(pid, rp)

    # ONNX エクスポート
    print(f'\n[{pid}] ONNX エクスポート中...')
    path = export_persona_onnx(policy, pid, rp)
    onnx_paths[pid] = path

    # GPU メモリ解放
    del policy
    torch.cuda.empty_cache()
    print(f'\n→ [{pid}] 完了。次のペルソナへ...\n')
    time.sleep(2)

print('='*60)
print('✓ 全ペルソナの学習・エクスポート完了')
print(f'保存先: マイドライブ > mesa_persona_onnx')
print()
for pid, path in onnx_paths.items():
    size_mb = os.path.getsize(path)/1e6
    print(f'  [{pid}] persona_{pid}.onnx  ({size_mb:.1f}MB)')

In [ ]:
# ============================================================
# セル 12: 最終確認
# ============================================================
print('mesa_persona_onnx フォルダの内容:')
print()
total_mb = 0
for fname in sorted(os.listdir(ONNX_DIR)):
    fpath = f'{ONNX_DIR}/{fname}'
    mb    = os.path.getsize(fpath)/1e6
    total_mb += mb
    has_data = os.path.exists(fpath+'.data')
    flag = '⚠️  外部データあり (ブラウザ非対応)' if has_data else '✓ 単一ファイル'
    print(f'  {fname:<42} {mb:6.1f}MB  {flag}')

print(f'\n  合計: {total_mb:.1f}MB')
print()
print('次のステップ:')
print('  1. Google Drive から .onnx と _meta.json をダウンロード')
print('  2. step3_persona_city_sim.html をブラウザで開く')
print('  3. 各ペルソナの .onnx と _meta.json を同時に選択してロード')
print('  4. Start Simulation !')